# Task 2 — Validate: count NEW rows by timestamp

**The simple idea:** the job runs daily, and the only question is — *did new data arrive in the last `window_minutes` minutes?*

This task counts bronze rows with `ingested_at >= now() - window_minutes`, publishes **`row_count` as a task value**, and the If/Else condition task compares it against 0.

> Example: `window_minutes = 10` → "any rows ingested in the last 10 minutes?" Rows from yesterday's run are outside the window → row_count = 0 → the notify branch runs.

In [0]:
# CELL 1 — Parameters
# catalog       : demo catalog (passed by the job)
# window_minutes: how far back to look for new rows (passed by the job).
#                 Example: 10 -> "any rows ingested in the last 10 minutes?"
# job_id/run_id : the run's real identifiers (passed by the job) — for the audit log
dbutils.widgets.text("catalog", "demo1", "Catalog that holds all demo tables")
dbutils.widgets.text("window_minutes", "10", "Look-back window in minutes")
dbutils.widgets.text("job_id", "interactive", "Job ID (passed by the job)")
dbutils.widgets.text("run_id", "interactive", "Run ID (passed by the job)")

catalog = dbutils.widgets.get("catalog")
window_minutes = int(dbutils.widgets.get("window_minutes"))
job_id = dbutils.widgets.get("job_id")
run_id = dbutils.widgets.get("run_id")

BRONZE_TABLE = f"{catalog}.bronze.sales_orders"
print(f"catalog        : {catalog}")
print(f"window_minutes : {window_minutes}")
print(f"job_id / run_id: {job_id} / {run_id}")

In [0]:
# CELL 2 — Count the NEW rows (ingested within the last N minutes)
# WHAT IS HAPPENING:
#   * One simple timestamp rule decides everything:
#         ingested_at >= now() - window_minutes
#   * Yesterday's rows are outside the window (job runs daily),
#     so row_count only reflects data that arrived recently.
from pyspark.sql import functions as F

new_rows_df = spark.table(BRONZE_TABLE).where(
    F.col("ingested_at") >= F.current_timestamp() - F.expr(f"INTERVAL {window_minutes} MINUTES")
)
row_count = new_rows_df.count()

print(f"Rows ingested in the last {window_minutes} minutes: {row_count}")
display(new_rows_df)

In [0]:
# CELL 3 — Publish row_count as a TASK VALUE (the condition task's input)
# The If/Else condition task 'check_row_count' evaluates:
#     {{tasks.validate.values.row_count}} > 0
data_available = row_count > 0
dbutils.jobs.taskValues.set(key="row_count", value=int(row_count))
dbutils.jobs.taskValues.set(key="data_available", value="true" if data_available else "false")

print(f"row_count      = {row_count}")
print(f"data_available = {data_available}")

from datetime import datetime
log_df = spark.createDataFrame(
    [(datetime.now(), job_id, run_id, "validate", int(row_count),
      "DATA_AVAILABLE" if data_available else "NO_DATA",
      f"rows ingested in last {window_minutes} minutes")],
    "logged_at TIMESTAMP, job_id STRING, run_id STRING, task_name STRING, "
    "rows_processed BIGINT, status STRING, message STRING",
)
log_df.write.mode("append").saveAsTable(f"{catalog}.bronze.job_run_log")
print("Audit row written.")

## Task 2 — recap

* `row_count` = number of rows ingested in the last `window_minutes` minutes — published as a task value (task run page → **Output values**).
* Next: the condition task `check_row_count` evaluates `row_count > 0` **without any compute**: TRUE → `transform_publish` (silver, SCD1) → `gold_kpi`, FALSE → `notify_no_data`.